In [ ]:
from msprep import MultiStatePrep
import numpy as np
import torch
from dgmsm.utils import set_global_seed
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


seed = 42
set_global_seed(seed)

msp = MultiStatePrep.load(f"data/commpass_meta.json")


msp_train, msp_val, msp_test = msp.split(
    test_size=0.2,
    val_size=0.2,
    stratify_by_state=True,
    random_state=seed,
    shuffle=True,
)


In [ ]:
from dgmsm import ModelConfig

cfg = ModelConfig(
    tmat=msp.tmat,
    base_ftypes=msp.baseline_ftypes,
    state_ftypes=msp.state_ftypes,
    n_time=20,
    z_dim=4,
    s_dim=2,
    y_dim=2,
    h_dim=8,
    f_dims=[16],
    f_latent=8,
    dropout=0.2,
    patience=10,
    aux_weight=0.2,
    batch_size=64,
    lr_decay_rate=0.01,
    weight_decay=0.01,
)
print(cfg)

In [ ]:

from dgmsm.utils import build_dataloader, MultiStateData, LabelTrafo

trafo = LabelTrafo(n_time=cfg.n_time)

states_train_disc = trafo.fit_transform(msp_train.states)
states_val_disc  = trafo.transform(msp_val.states)
states_test_disc = trafo.transform(msp_test.states)

train_loader = build_dataloader(
    states_df=states_train_disc,
    baseline_df=msp_train.baseline,
    batch_size=cfg.batch_size,
    shuffle=True,
    **cfg.dataset_kwargs(),
)


val_loader = build_dataloader(
    states_df=states_val_disc,
    baseline_df=msp_val.baseline,
    batch_size=cfg.batch_size,
    **cfg.dataset_kwargs(),
)

test_dataset = MultiStateData(
    states_df=states_test_disc,
    baseline_df=msp_test.baseline,
    **cfg.dataset_kwargs(),
)

from dgmsm.hivae.normalization import compute_norm_params_from_dataloader

base_norm_params, state_norm_params = compute_norm_params_from_dataloader(
    config=cfg,
    dataloader=train_loader,
    device=device,
)

model = cfg.build_model(
    base_norm_params=base_norm_params,
    state_norm_params=state_norm_params,
)

history = model.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    device=device, 
    restore_best=True,
    verbose=True,
    **cfg.training_kwargs()
)


---
### State specific sojourn times

In [ ]:
eval = model.eval_surv(dataset=test_dataset)

---
### Sampling from the prior distribution

In [ ]:
from dgmsm.synthetic_msp import prior_sample_msp
from msprep.utils.syndat_scores import syndat_scores


synthetic_msp = prior_sample_msp(
    model=model,
    msp_template=msp.get_template(),
    n=msp_train.n_persons,
    time_delta=trafo.time_delta,
)

msp_train.dataset_name = "Training Data"

df_scores = syndat_scores(
    real_msp=msp_train,
    synthetic_msp=synthetic_msp,
    stratify_by_visit=True,
)
df_scores

In [ ]:
from dgmsm.vt_eval.multi_state_landmark import multi_state_landmark, plot_multi_state_landmark


result = multi_state_landmark(
    model, 
    test_dataset, 
    trafo, 
    msp.state_names,
    n_sim=1000, 
    device=device, 
    seed=seed,
)

figures = plot_multi_state_landmark(result)